In [1]:
import pandas as pd
import numpy as np
import anndata as ad
import scanpy as sc
import matplotlib.pyplot as plt

In [6]:
PRT_names = ['APE','COL','ILE','PROS','STO']
PRTs = []
for name in PRT_names:
    sm = sc.read(f'PRT_{name}_SM.h5ad')
    sm.obs['phenotype'] = sm.obs['phenotype'].str.replace('likely-', '').astype('category')

    # Define your mapping from fine to broad phenotype
    phenotype_mapping = {
        'Stroma': 'Mesenchymal-Stromal',
        'Endothelial': 'Endothelial',
        'Epithelial': 'Epithelial',
        'Myeloid': 'Myeloid',
        'Immune': 'Lymphoid-Other',
        'Dendritic Cells': 'Lymphoid-DC',
        'T Cells': 'Lymphoid-T',
        'B Cells': 'Lymphoid-B',
    }

    # Apply the mapping
    sm.obs['phenotype_broad'] = sm.obs['phenotype'].map(phenotype_mapping)

    PRT = sc.read(f'../SAHAMaxFuse/PRT_{name}.h5ad')
    PRT.obs['phenotype'] = sm.obs['phenotype'].astype('category')
    PRT.obs['phenotype_broad'] = sm.obs['phenotype_broad'].astype('category')

    PRTs.append(PRT)
PRT = ad.concat(PRTs, uns_merge='first')
PRT

AnnData object with n_obs × n_vars = 1164666 × 64
    obs: 'Area', 'CenterX_global_px', 'CenterY_global_px', 'cell_id', 'batch', 'SAHA_name', 'leiden', 'leiden_0.5', 'leiden_1', 'celltype_mf', 'phenotype', 'phenotype_broad'
    uns: 'SAHA_name_colors', 'leiden', 'leiden_0.5_colors', 'leiden_1_colors', 'neighbors', 'pca', 'umap'
    obsm: 'X_harmony', 'X_pca', 'X_pca_harmony', 'X_umap', 'X_umap_after_harmony', 'X_umap_before_harmony', 'spatial'
    layers: 'normalized', 'raw'

In [ ]:
# Crosstab
table = pd.crosstab(PRT.obs['celltype_mf'], PRT.obs['phenotype_broad'])

# Force string-based sorting
table.columns = table.columns.astype(str)
table = table[sorted(table.columns)]
table = table.sort_index(axis=0)

print(table)

phenotype_broad      Endothelial  Epithelial  Lymphoid-B  Lymphoid-DC  \
celltype_mf                                                             
Endothelial                 5996        2769        2585         1759   
Epithelial                 42609      275425       12899         6273   
Erythroid                     97        5193          97          419   
Immune-Other                 126          88         332         2046   
Lymphoid-B                   557        1071       61710          933   
Lymphoid-DC                  659        1287        4900         3281   
Lymphoid-Other             11185       32846       52426        18744   
Lymphoid-Plasma            11642       18674        8803         9083   
Lymphoid-T                   820        3713       21152          691   
Mesenchymal-Stromal        11133       11689        4176         2396   
Myeloid                     1218        2027        1144         2705   
Neuronal-Glial              1677         559       

In [9]:
table.to_csv("SM_MF.csv")
